In [1]:
# 各初期化手法を実行する前に実行するコード
BASE_MODEL = "cl-tohoku/bert-base-japanese-v3"
ADDED_MODEL = "tmbert_bf_randinit"

f:\RikiResearch\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


## ランダムな初期化の後に正規化

In [3]:
from transformers import BertForPreTraining, BertJapaneseTokenizer
import torch

OUTPUT_MODEL = "tmbert_bf_randinit_norm"

from transformers import BertJapaneseTokenizer, BertForPreTraining

base_tokenizer: BertJapaneseTokenizer = BertJapaneseTokenizer.from_pretrained(BASE_MODEL)
added_tokenizer: BertJapaneseTokenizer = BertJapaneseTokenizer.from_pretrained(ADDED_MODEL)

base_model: BertForPreTraining = BertForPreTraining.from_pretrained(BASE_MODEL)
added_model: BertForPreTraining = BertForPreTraining.from_pretrained(ADDED_MODEL)

base_vocab_size = len(base_tokenizer)
added_vocab_size = len(added_tokenizer)

with torch.no_grad():
    base_vocab_num = len(base_tokenizer)

    base_mean_norm = 0.0
    for i in range(base_vocab_num):
        base_mean_norm += added_model.bert.embeddings.word_embeddings.weight[i].norm()
    base_mean_norm /= base_vocab_num

    print(base_mean_norm)
    new_vocab_num = len(added_tokenizer)
    for i in range(base_vocab_num, new_vocab_num):
        temp_norm = added_model.bert.embeddings.word_embeddings.weight[i].norm()
        added_model.bert.embeddings.word_embeddings.weight[i] = added_model.bert.embeddings.word_embeddings.weight[i] / temp_norm
    
    print(added_model.bert.embeddings.word_embeddings.weight[-1].norm())

    added_model.save_pretrained(OUTPUT_MODEL)
    added_tokenizer.save_pretrained(OUTPUT_MODEL)

tensor(1.4544)
tensor(1.)


## 伝統医学のようなベクトルをベースにした初期化

In [4]:
OUTPUT_MODEL = "tmbert_bf_tmvec"

from transformers import BertJapaneseTokenizer, BertForPreTraining

base_tokenizer: BertJapaneseTokenizer = BertJapaneseTokenizer.from_pretrained(BASE_MODEL)
added_tokenizer: BertJapaneseTokenizer = BertJapaneseTokenizer.from_pretrained(ADDED_MODEL)

base_model: BertForPreTraining = BertForPreTraining.from_pretrained(BASE_MODEL)
added_model: BertForPreTraining = BertForPreTraining.from_pretrained(ADDED_MODEL)

base_vocab_size = len(base_tokenizer)
added_vocab_size = len(added_tokenizer)

from transformers import BertForPreTraining, BertJapaneseTokenizer, BertModel, set_seed
import numpy as np
import torch
from torch.nn.functional import cosine_similarity


with torch.no_grad():
    sep_str = "・"
    # 以下のようにリストにすることも可能
    # init_words = ["伝統医学", "中医学", "漢方"]
    init_words = ["伝統医学"]
    init_sentence = sep_str.join(init_words)
    print(init_sentence)
    sep_token_id = base_tokenizer.convert_tokens_to_ids(sep_str)
    input_ids = base_tokenizer.encode(init_sentence, return_tensors="pt")
    print(input_ids)
    output_ids = base_model.bert(input_ids)
    output_vec = output_ids.last_hidden_state[0]
    mean_tensor = torch.zeros(output_vec.shape[1])
    token_num = 0
    for i in range(output_vec.shape[0]-1):
        if input_ids[0][i] == sep_token_id:
            continue
        mean_tensor += output_vec[i]
        token_num+=1
    print(mean_tensor.shape)
    mean_tensor /= token_num


    random_mean_word_vec = torch.zeros(output_vec.shape[1])
    base_vocab_num = len(base_tokenizer)
    new_vocab_num = len(added_tokenizer)
    vocab_range = new_vocab_num - base_vocab_num

    base_norm_mean = 0.0
    for i in range(0, base_vocab_num):
        base_norm_mean += added_model.bert.embeddings.word_embeddings.weight[i].norm()
    base_norm_mean /= base_vocab_num

    for i in range(base_vocab_num, new_vocab_num):
        random_mean_word_vec += added_model.bert.embeddings.word_embeddings.weight[i]
    random_mean_word_vec /= vocab_range
    for i in range(base_vocab_num, new_vocab_num):
        rand_element = added_model.bert.embeddings.word_embeddings.weight[i] - random_mean_word_vec
        vec = (added_model.bert.embeddings.word_embeddings.weight[i] - random_mean_word_vec + mean_tensor)
        added_model.bert.embeddings.word_embeddings.weight[i] = (vec / vec.norm()) * base_norm_mean
    added_model.save_pretrained(OUTPUT_MODEL)
    added_tokenizer.save_pretrained(OUTPUT_MODEL)


伝統医学
tensor([[    2, 14218, 14254,     3]])
torch.Size([768])


## マスクトークンを利用した初期化

In [2]:
OUTPUT_MODEL = "tmbert_bf_maskvec"

from transformers import BertJapaneseTokenizer, BertForPreTraining

base_tokenizer: BertJapaneseTokenizer = BertJapaneseTokenizer.from_pretrained(BASE_MODEL)
added_tokenizer: BertJapaneseTokenizer = BertJapaneseTokenizer.from_pretrained(ADDED_MODEL)

base_model: BertForPreTraining = BertForPreTraining.from_pretrained(BASE_MODEL)
added_model: BertForPreTraining = BertForPreTraining.from_pretrained(ADDED_MODEL)

base_vocab_size = len(base_tokenizer)
added_vocab_size = len(added_tokenizer)

import json
from transformers import BertJapaneseTokenizer, BertModel, BertForPreTraining
import torch

CORPUS_LIST = [
    "../resources/corpus/chui_word_dic_corpus.txt",
    "../resources/corpus/sho_db_corpus.txt",
    "../resources/corpus/who_tcm_corpus.txt"
]

corpus_text_list = []

for corpus_path in  CORPUS_LIST:
    with open(corpus_path, mode="r", encoding="utf-8") as fp:
        lines = fp.readlines()
        corpus_text_list += lines


corpus_text_list = [
    text.replace("\n", "") for text in corpus_text_list if text != "\n" or text != ""
]

ids_list = [
    added_tokenizer.encode(text) for text in corpus_text_list
]

mask_token_list = [
    added_tokenizer.convert_ids_to_tokens(id) for id in range(base_vocab_size, added_vocab_size)
]

mask_ids_dict = {}
for token in mask_token_list:
    mask_ids_dict[token] = []
    id = added_tokenizer.convert_tokens_to_ids(token)
    for ids in ids_list:
        if id in ids:
            mask_ids_dict[token].append(ids)


gpu_device = torch.device("cuda:0")
cpu_device = torch.device("cpu")

def find_nan_indices(tensor):
    nan_mask = torch.isnan(tensor)
    nan_indices = torch.nonzero(nan_mask)
    return nan_indices

def has_nan(tensor):
    nan_mask = torch.isnan(tensor)
    has_nan = torch.any(nan_mask)
    return bool(has_nan)    


sentence_dict = {}
added_vocab_list = added_tokenizer.convert_ids_to_tokens([i for i in range(base_vocab_size, added_vocab_size)])

truncated_data_list = []

# [CLS]と[SEP]
seq_len = added_tokenizer.model_max_length - 2

mask_token_id: int = base_tokenizer.convert_tokens_to_ids("[MASK]")
mask_token_weight = base_model.bert.embeddings.word_embeddings.weight[mask_token_id]
count1 = 0
count2 = 0


with torch.no_grad():
    for token, ids_list in mask_ids_dict.items():
        id = added_tokenizer.convert_tokens_to_ids(token)
        new_embedding = torch.zeros(base_model.config.hidden_size)

        ids_tensor_list = [
            torch.tensor(ids, dtype=torch.int).unsqueeze(0) for ids in ids_list
        ]
        for ids_tensor in ids_tensor_list:
            mask_positions = (ids_tensor == id).nonzero(as_tuple=True)
            ids_tensor[ids_tensor == id] = mask_token_id
            outputs = added_model.bert(input_ids=ids_tensor, attention_mask=torch.ones_like(ids_tensor))
            last_hidden_states = outputs.last_hidden_state
            replaced_embeddings = last_hidden_states[mask_positions]
            embedding_num = replaced_embeddings.shape[0]
            if embedding_num == 0:
                continue

            temp_new_embedding = torch.zeros(base_model.config.hidden_size)
            for i in range(embedding_num):
                temp_new_embedding += (replaced_embeddings[i] - mask_token_weight)
            temp_new_embedding /= embedding_num
            new_embedding += temp_new_embedding
        
        ids_list_len = len(ids_tensor_list)
        new_embedding /= ids_list_len
        added_model.bert.embeddings.word_embeddings.weight[id] = new_embedding

    mean_norm = 0.0
    for i in range(base_vocab_size):
        mean_norm += base_model.bert.embeddings.word_embeddings.weight[i].to(cpu_device).norm()
    mean_norm /= base_vocab_size


    for i in range(base_vocab_size, added_vocab_size):
        vec = added_model.bert.embeddings.word_embeddings.weight[i]
        added_model.bert.embeddings.word_embeddings.weight[i, :] = (vec / vec.norm()) * mean_norm


    added_model.save_pretrained(OUTPUT_MODEL)
    added_tokenizer.save_pretrained(OUTPUT_MODEL)
        

RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation.